# ASAGI material NetCDF -> ParaView VTU on Google Colab

Converts an ASAGI material file (e.g. `safs_material_cvm_s4.26.m01.nc`, the
output of the velocity->NetCDF notebook) into a `.vtu` you can open in
**ParaView** to inspect `rho / mu / lambda` and the derived `Vs / Vp / Poisson`
in 3-D. We run the project's `nc_to_vtu.py` and preview a slice inline.

### Before you start - put the inputs on Google Drive
```
<your_case>/safs_material_cvm_s4.26.m01.nc   (the ASAGI material NetCDF)
<somewhere>/nc_to_vtu.py                      (this tool; auto-found near the
                                               .nc, or set SCRIPT in step 3)
```
The `.vtu` is written next to the `.nc`. ParaView runs on your **own computer**
(Colab can't display it) -- so keep the file on Drive or download it (step 6).

## What it does (background)
- Reads the NetCDF4 material file with **h5py** -- NetCDF4 *is* HDF5, so no
  `netCDF4` / `vtk` / `meshio` needed: axes `x, y, z` (metres) plus a compound
  `data(z, y, x)` of `rho, mu, lambda`.
- Builds a hexahedral grid (`VTK_HEXAHEDRON`, not VOXEL, for tool
  compatibility) with the values as **point data** on the nodes, so ParaView
  interpolates linearly -- exactly like easi/ASAGI `interpolation: linear`.
- Stores raw `rho_kg_m3, mu_GPa, lambda_GPa` plus derived
  `Vs_m_s, Vp_m_s, poisson_ratio` (skip with `NO_DERIVED = True`).

**Size:** the full SAFS grid (264 x 195 x 181) is a ~660 MB VTU. `STRIDE = N`
subsamples every Nth node per axis (`STRIDE = 2` is ~1/8 the size); combine
with `NO_DERIVED = True` for an even lighter file.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Dependencies
Nothing to install: **h5py**, **numpy** and **matplotlib** all ship with
Colab. (`nc_to_vtu.py` deliberately uses only h5py + numpy -- it reads the
NetCDF through HDF5 and writes the VTU's raw binary by hand.)

In [ ]:
import h5py, numpy, matplotlib
print("h5py", h5py.__version__, "| numpy", numpy.__version__,
      "| matplotlib", matplotlib.__version__)

## 3. Point at your data on Drive
**Edit `NC`** to your ASAGI material file. `SCRIPT` can stay `""` -- the
notebook auto-finds `nc_to_vtu.py` near the `.nc` (upload it there, or set the
full path). `STRIDE` / `NO_DERIVED` control the output size (see background).
Paths and the output name are checked with asserts so mistakes fail here.

In [ ]:
import os, glob

NC     = "/content/drive/MyDrive/safs_seisol_v2_2_0_LSW_s4.26.m01cvm/safs_material_cvm_s4.26.m01.nc"  # <-- EDIT
SCRIPT = ""           # "" = auto-find nc_to_vtu.py near the .nc; else full path
OUT_DIR    = os.path.dirname(NC)   # write the .vtu next to the .nc (on Drive)
STRIDE     = 1        # 1 = full grid (~660 MB); 2 ~= 1/8 the size
NO_DERIVED = False    # True = skip Vs / Vp / poisson_ratio (smaller file)

_d = os.path.dirname(NC)
_cands = [os.path.join(_d, "nc_to_vtu.py"),
          os.path.join(_d, "nc_to_vtu", "nc_to_vtu.py"),
          os.path.join(_d, "toolbox", "nc_to_vtu", "nc_to_vtu.py")]
if not SCRIPT:
    SCRIPT = next((c for c in _cands if os.path.exists(c)), "")

assert os.path.exists(NC), f"{NC} not found - fix NC"
assert SCRIPT, ("nc_to_vtu.py not found near "
                f"{_d} - upload it there or set SCRIPT to its full path")

# the tool names the output <stem>[_s{STRIDE}].vtu in OUT_DIR
_stem = os.path.splitext(os.path.basename(NC))[0] + (f"_s{STRIDE}" if STRIDE > 1 else "")
VTU   = os.path.join(OUT_DIR, _stem + ".vtu")

print("input  :", NC, f"({os.path.getsize(NC)/1e6:.0f} MB)")
print("tool   :", SCRIPT)
print("output :", VTU, f"(stride {STRIDE}, derived={'no' if NO_DERIVED else 'yes'})")

## 4. Convert NetCDF -> VTU
We run `nc_to_vtu.py` with `sys.executable` (guarantees the right Python; no
`!python` PATH surprises). The tool prints the grid size and the min/max of
every field; the `.vtu` lands in `OUT_DIR`. Large strides finish in seconds;
the full grid takes longer and writes a big file to Drive.

In [ ]:
import subprocess, sys

cmd = [sys.executable, SCRIPT, NC, "--out-dir", OUT_DIR, "--stride", str(STRIDE)]
if NO_DERIVED:
    cmd += ["--no-derived"]
print("running:", " ".join(cmd), "\n", flush=True)
subprocess.run(cmd, check=True)      # streams the tool's progress live

assert os.path.exists(VTU), f"expected {VTU} but it is missing"
print(f"\nwrote {VTU}  ({os.path.getsize(VTU)/1e6:.0f} MB, on your Drive)")

## 5. Preview a slice inline
A quick look before hauling the file into ParaView: `Vs = sqrt(mu/rho)` across
the model at the surface, and its depth profile at the centre (read straight
from the `.nc` at the same `STRIDE` the VTU used).

In [ ]:
import numpy as np, matplotlib.pyplot as plt

with h5py.File(NC, "r") as f:
    xs = f["x"][::STRIDE]; ys = f["y"][::STRIDE]; zs = f["z"][::STRIDE]
    data = f["data"][::STRIDE, ::STRIDE, ::STRIDE]   # compound (rho, mu, lambda)
    rho = data["rho"].astype(float); mu = data["mu"].astype(float)

Vs = np.sqrt(mu / rho)               # (nz, ny, nx)
jc, ic = len(ys) // 2, len(xs) // 2

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
pc = ax[0].pcolormesh(xs / 1e3, ys / 1e3, Vs[-1], cmap="viridis", shading="auto")
ax[0].set_aspect("equal"); ax[0].set_title("Vs at surface  [m/s]")
ax[0].set_xlabel("UTM x [km]"); ax[0].set_ylabel("UTM y [km]")
fig.colorbar(pc, ax=ax[0], shrink=0.85)

ax[1].plot(Vs[:, jc, ic], zs / 1e3)
ax[1].set_xlabel("Vs [m/s]"); ax[1].set_ylabel("elevation z [km]")
ax[1].set_title("Vs depth profile (centre)"); ax[1].grid(True)
plt.tight_layout(); plt.show()

## 6. Get the VTU into ParaView
The file is already on your Drive (in `OUT_DIR`). Either open that Drive folder
on your computer, or set `DOWNLOAD = True` to stream it through the browser
(can be large). Then in ParaView: **File > Open**, and colour by `Vs_m_s`,
`mu_GPa`, `poisson_ratio`, ...

In [ ]:
DOWNLOAD = False     # True = stream the .vtu to your computer (may be large)

if DOWNLOAD:
    from google.colab import files
    print("downloading", VTU, "...")
    files.download(VTU)
else:
    print("VTU left on Drive:", VTU)
    print("Open in ParaView (desktop): File > Open -> colour by Vs_m_s / mu_GPa / ...")

## Notes / caveats
- **ParaView is desktop-only here** -- Colab can't render the VTU. The point of
  running on Colab is to offload the conversion (and keep the big file on
  Drive); you open it locally.
- **File size:** full grid ~660 MB. `STRIDE = 2` -> ~1/8; `NO_DERIVED = True`
  drops the 3 derived fields. Re-running step 4 overwrites the `.vtu`.
- **Point data + hexahedra** = linear interpolation in ParaView, matching
  easi/ASAGI `interpolation: linear`. Uses `VTK_HEXAHEDRON` (12), not VOXEL
  (11), which some readers reject.
- Reads NetCDF4 via **h5py** (NetCDF4 is HDF5). A file with a compound `data`
  var or with separate `rho`/`mu`/`lambda` datasets both work.
- To convert another material file, change `NC` (step 3) and re-run steps 3-6.